In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
!pip uninstall -y -q torchao
!pip install -q peft

import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "60"

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

train_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
test_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
sub_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

options = ["A", "B", "C", "D", "E"]
label_map = {opt: i for i, opt in enumerate(options)}

train["label"] = train["answer"].map(label_map)

print("Q1:", train.loc[150, "label"])

row0 = train.iloc[0]
formatted_b = str(row0["prompt"]) + " [SEP] " + str(row0["B"])
print("Q2:", len(formatted_b))

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def build_choices(row):
    prompt = str(row["prompt"])
    return [prompt + " [SEP] " + str(row[opt]) for opt in options]

choices0 = build_choices(row0)
enc0 = tokenizer(choices0, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
input_ids0 = enc0["input_ids"].view(1, 5, 128)
print("Q3:", input_ids0.shape[1])

batch16 = train.iloc[:16]
flat_choices = []
for _, r in batch16.iterrows():
    flat_choices.extend(build_choices(r))

enc16 = tokenizer(flat_choices, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
input_ids16 = enc16["input_ids"].view(16, 5, 128)
print("Q4:", input_ids16.numel())

try:
    model = AutoModelForMultipleChoice.from_pretrained(model_name)
except Exception:
    model = AutoModelForMultipleChoice.from_pretrained(model_name, use_safetensors=False)
model.eval()

with torch.no_grad():
    out = model(input_ids=input_ids0, attention_mask=enc0["attention_mask"].view(1, 5, 128))
print("Q5:", out.logits.shape[1])

label0 = torch.tensor([row0["label"]])
with torch.no_grad():
    out_loss = model(input_ids=input_ids0, attention_mask=enc0["attention_mask"].view(1, 5, 128), labels=label0)
print("Q6:", out_loss.loss.dim())

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
lora_model = get_peft_model(model, lora_cfg)
trainable = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print("Q7:", trainable)

def tokenize_row(row):
    choices = build_choices(row)
    enc = tokenizer(choices, padding="max_length", truncation=True, max_length=128)
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": row["label"],
    }

subset100 = train.iloc[:100].reset_index(drop=True)
records = [tokenize_row(subset100.iloc[i]) for i in range(len(subset100))]
ds100 = Dataset.from_list(records)
print("Q8:", len(ds100[0]["input_ids"]))

def tokenize_row_small(row, max_len):
    choices = build_choices(row)
    enc = tokenizer(choices, padding="max_length", truncation=True, max_length=max_len)
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": row["label"],
    }

subset32 = train.iloc[:32].reset_index(drop=True)
records32 = [tokenize_row_small(subset32.iloc[i], 64) for i in range(len(subset32))]
ds32 = Dataset.from_list(records32)
ds32.set_format(type="torch")

training_args = TrainingArguments(
    output_dir="./mcq_lora_out",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    report_to="none",
    save_strategy="no",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=ds32,
)

train_result = trainer.train()
print("Q9:", trainer.state.global_step)

lora_model.eval()
choices0_small = build_choices(row0)
enc0_small = tokenizer(choices0_small, padding="max_length", truncation=True, max_length=64, return_tensors="pt")
input_ids0_small = enc0_small["input_ids"].view(1, 5, 64)
attn0_small = enc0_small["attention_mask"].view(1, 5, 64)

with torch.no_grad():
    final_out = lora_model(input_ids=input_ids0_small, attention_mask=attn0_small)

probs = F.softmax(final_out.logits, dim=1)
prob_e = probs[0][4].item()
print("Q10:", round(prob_e, 4))

Q1: 2
Q2: 407
Q3: 5
Q4: 10240


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Q5: 5
Q6: 0
Q7: 295681
Q8: 5


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.634642
2,1.624049
3,1.555494
4,1.704571


Q9: 4
Q10: 0.1957
